# Malicious Traffic Pattern Clustering Using K-Means and Hierarchical Clustering

**Dataset:** NSL-KDD (`KDDTrain+.txt` / `KDDTrain__1_.csv`)
**Techniques:** K-Means Clustering, Hierarchical (Agglomerative) Clustering
**Evaluation:** Elbow Method, Silhouette Score, Davies-Bouldin Index, Cluster Purity, Adjusted Rand Index (vs. known attack labels)

This notebook is written to run top-to-bottom in Google Colab. Run the setup cell first, upload your dataset when prompted, then run every cell in order.

## 1. Setup — folders and libraries

In [ ]:
# Create the project folder structure inside Colab's runtime
import os

for folder in ["data", "results", "results/figures"]:
    os.makedirs(folder, exist_ok=True)

print("Folders ready:", os.listdir("."))

In [ ]:
!pip install -q scikit-learn scipy pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    adjusted_rand_score,
)
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_style("whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Upload the dataset

Run the cell below, then choose your CSV (e.g. `KDDTrain+.csv`) when the upload button appears.
It will be saved into the `data/` folder automatically.

In [ ]:
from google.colab import files

uploaded = files.upload()  # opens a file picker in Colab

for fname in uploaded.keys():
    dest = os.path.join("data", fname)
    os.rename(fname, dest)
    print(f"Saved to {dest}")

> **Alternative — Google Drive:** if your dataset already lives in Drive instead of on your machine, use this instead of the upload cell:
> ```python
> from google.colab import drive
> drive.mount('/content/drive')
> import shutil
> shutil.copy('/content/drive/MyDrive/<path-to-your-file>.csv', 'data/KDDTrain.csv')
> ```

## 3. Load the dataset

In [ ]:
DATA_DIR = "data"

csv_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("No CSV found in data/. Run the upload cell above first.")

DATA_PATH = os.path.join(DATA_DIR, csv_files[0])
print("Using dataset:", DATA_PATH)

# NSL-KDD has 41 features + label + difficulty score, with no header row in the
# original KDDTrain+.txt release. If your file already has a header row (as in
# KDDTrain__1_.csv), pandas will pick it up automatically; otherwise we assign
# the standard NSL-KDD column names.
NSL_KDD_COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label", "difficulty",
]

first_cell = pd.read_csv(DATA_PATH, nrows=1)
has_header = "protocol_type" in first_cell.columns or "label" in first_cell.columns

if has_header:
    df = pd.read_csv(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH, names=NSL_KDD_COLUMNS)

df.columns = [c.strip() for c in df.columns]
print("Shape:", df.shape)
df.head()

## 4. Quick EDA

In [ ]:
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDuplicate rows:", df.duplicated().sum())

print("\nLabel distribution (top 15):")
print(df["label"].value_counts().head(15))

plt.figure(figsize=(10, 5))
df["label"].value_counts().head(15).plot(kind="bar")
plt.title("Top 15 traffic labels")
plt.ylabel("Count")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.savefig("results/figures/label_distribution.png", dpi=150)
plt.show()

## 5. Preprocessing

Steps: derive a binary `attack_type` (normal vs. attack) for later evaluation, encode
categorical features, drop non-feature columns (`label`, `difficulty`), scale numeric
features, and remove constant columns.

In [ ]:
df_clean = df.copy()

# Keep the original multi-class label and a binary version for evaluation later
df_clean["attack_type"] = df_clean["label"].apply(lambda x: "normal" if str(x).strip() == "normal" else "attack")

# Encode the three categorical features
categorical_cols = ["protocol_type", "service", "flag"]
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    encoders[col] = le

# Separate features from labels
drop_cols = [c for c in ["label", "difficulty", "attack_type"] if c in df_clean.columns]
X = df_clean.drop(columns=drop_cols)
y_multiclass = df_clean["label"]
y_binary = df_clean["attack_type"]

# Numeric-only safety net + handle inf/NaN
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

# Drop constant columns (zero variance -> no clustering signal)
nunique = X.nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
if constant_cols:
    print("Dropping constant columns:", constant_cols)
    X = X.drop(columns=constant_cols)

print("Final feature matrix shape:", X.shape)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 6. Dimensionality reduction (PCA)

Used purely for 2D visualization of the clusters, not for clustering itself.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained variance (PC1, PC2): {pca.explained_variance_ratio_}")
print(f"Total variance captured: {pca.explained_variance_ratio_.sum():.2%}")

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=3, alpha=0.3)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA projection of network traffic")
plt.tight_layout()
plt.savefig("results/figures/pca_projection.png", dpi=150)
plt.show()

## 7. Choosing the number of clusters (K-Means)

We evaluate K = 2..8 using inertia (elbow method) and silhouette score.
For speed on large datasets, silhouette is computed on a random sample.

In [ ]:
SAMPLE_SIZE_FOR_SILHOUETTE = 10000  # keeps silhouette scoring fast on large datasets
rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(len(X_scaled), size=min(SAMPLE_SIZE_FOR_SILHOUETTE, len(X_scaled)), replace=False)

k_range = range(2, 9)
inertias, silhouettes = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled[sample_idx], labels[sample_idx])
    silhouettes.append(sil)
    print(f"k={k}: inertia={km.inertia_:.1f}, silhouette={sil:.4f}")

metrics_df = pd.DataFrame({"k": list(k_range), "inertia": inertias, "silhouette": silhouettes})
metrics_df.to_csv("results/kmeans_metrics.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(metrics_df["k"], metrics_df["inertia"], marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")

axes[1].plot(metrics_df["k"], metrics_df["silhouette"], marker="o", color="darkorange")
axes[1].set_title("Silhouette Score vs k")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette score")

plt.tight_layout()
plt.savefig("results/figures/k_selection.png", dpi=150)
plt.show()

best_k = int(metrics_df.loc[metrics_df["silhouette"].idxmax(), "k"])
print("\nSelected k (highest silhouette):", best_k)

## 8. Final K-Means clustering

In [ ]:
kmeans_final = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans_final.fit_predict(X_scaled)

kmeans_silhouette = silhouette_score(X_scaled[sample_idx], kmeans_labels[sample_idx])
kmeans_db = davies_bouldin_score(X_scaled[sample_idx], kmeans_labels[sample_idx])

print(f"K-Means (k={best_k}) — Silhouette: {kmeans_silhouette:.4f}, Davies-Bouldin: {kmeans_db:.4f}")

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap="tab10", s=3, alpha=0.4)
plt.title(f"K-Means clusters (k={best_k}) — PCA projection")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.savefig("results/figures/kmeans_clusters.png", dpi=150)
plt.show()

## 9. Hierarchical (Agglomerative) Clustering

Agglomerative clustering is O(n²) in memory/time, so it's run on a random sample
rather than the full dataset (standard practice for large network-flow datasets).

In [ ]:
HIER_SAMPLE_SIZE = 5000
hier_idx = rng.choice(len(X_scaled), size=min(HIER_SAMPLE_SIZE, len(X_scaled)), replace=False)
X_hier_sample = X_scaled[hier_idx]

hier_model = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
hier_labels_sample = hier_model.fit_predict(X_hier_sample)

hier_silhouette = silhouette_score(X_hier_sample, hier_labels_sample)
hier_db = davies_bouldin_score(X_hier_sample, hier_labels_sample)

print(f"Hierarchical (k={best_k}, sample n={HIER_SAMPLE_SIZE}) — "
      f"Silhouette: {hier_silhouette:.4f}, Davies-Bouldin: {hier_db:.4f}")

# PCA view of the same sample for a fair visual comparison
X_pca_sample = pca.transform(X_hier_sample)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca_sample[:, 0], X_pca_sample[:, 1], c=hier_labels_sample, cmap="tab10", s=6, alpha=0.6)
plt.title(f"Hierarchical clusters (k={best_k}) — PCA projection (sample)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.savefig("results/figures/hierarchical_clusters.png", dpi=150)
plt.show()

In [ ]:
# Dendrogram — run on a smaller sub-sample for a readable plot
DENDRO_SAMPLE_SIZE = 300
dendro_idx = rng.choice(len(X_hier_sample), size=min(DENDRO_SAMPLE_SIZE, len(X_hier_sample)), replace=False)

linkage_matrix = linkage(X_hier_sample[dendro_idx], method="ward")

plt.figure(figsize=(12, 6))
dendrogram(linkage_matrix, truncate_mode="lastp", p=30, leaf_rotation=90)
plt.title("Hierarchical Clustering Dendrogram (sample)")
plt.xlabel("Sample index / cluster size")
plt.ylabel("Distance")
plt.tight_layout()
plt.savefig("results/figures/dendrogram.png", dpi=150)
plt.show()

## 10. Evaluation against known attack labels

Since NSL-KDD is actually labeled, we can validate how well the *unsupervised*
clusters line up with ground truth — cluster purity, Adjusted Rand Index (ARI),
and a crosstab against the binary normal/attack label.

In [ ]:
def cluster_purity(cluster_labels, true_labels):
    df_tmp = pd.DataFrame({"cluster": cluster_labels, "true": true_labels})
    majority = df_tmp.groupby("cluster")["true"].agg(lambda s: s.value_counts().iloc[0])
    counts = df_tmp.groupby("cluster")["true"].agg(lambda s: s.value_counts().max())
    return counts.sum() / len(df_tmp)

# --- K-Means evaluation (full dataset) ---
kmeans_purity = cluster_purity(kmeans_labels, y_binary.values)
kmeans_ari = adjusted_rand_score(y_binary.values, kmeans_labels)

print("=== K-Means vs. ground truth (normal/attack) ===")
print(f"Purity: {kmeans_purity:.4f}")
print(f"Adjusted Rand Index: {kmeans_ari:.4f}")
print()
print(pd.crosstab(kmeans_labels, y_binary.values, rownames=["Cluster"], colnames=["True label"]))

# --- Hierarchical evaluation (on the same sample it was fit on) ---
hier_purity = cluster_purity(hier_labels_sample, y_binary.values[hier_idx])
hier_ari = adjusted_rand_score(y_binary.values[hier_idx], hier_labels_sample)

print("\n=== Hierarchical vs. ground truth (normal/attack, sample) ===")
print(f"Purity: {hier_purity:.4f}")
print(f"Adjusted Rand Index: {hier_ari:.4f}")
print()
print(pd.crosstab(hier_labels_sample, y_binary.values[hier_idx], rownames=["Cluster"], colnames=["True label"]))

## 11. Side-by-side comparison summary

In [ ]:
comparison = pd.DataFrame({
    "Method": ["K-Means", "Hierarchical"],
    "k": [best_k, best_k],
    "Silhouette Score": [kmeans_silhouette, hier_silhouette],
    "Davies-Bouldin Index": [kmeans_db, hier_db],
    "Cluster Purity": [kmeans_purity, hier_purity],
    "Adjusted Rand Index": [kmeans_ari, hier_ari],
})

comparison.to_csv("results/algorithm_comparison.csv", index=False)
comparison

## 12. Save cluster assignments

In [ ]:
results_df = pd.DataFrame({
    "kmeans_cluster": kmeans_labels,
    "true_label": y_multiclass.values,
    "attack_type": y_binary.values,
})
results_df.to_csv("results/kmeans_cluster_assignments.csv", index=False)

hier_results_df = pd.DataFrame({
    "hierarchical_cluster": hier_labels_sample,
    "true_label": y_multiclass.values[hier_idx],
    "attack_type": y_binary.values[hier_idx],
})
hier_results_df.to_csv("results/hierarchical_cluster_assignments.csv", index=False)

print("Saved:")
print(" - results/kmeans_metrics.csv")
print(" - results/kmeans_cluster_assignments.csv")
print(" - results/hierarchical_cluster_assignments.csv")
print(" - results/algorithm_comparison.csv")
print(" - results/figures/*.png")

## 13. (Optional) Download results as a zip

Run this to bundle everything in `results/` into a single zip you can download
from Colab's file browser.

In [ ]:
import shutil
shutil.make_archive("clustering_results", "zip", "results")

from google.colab import files
files.download("clustering_results.zip")